# cAPTure: one-seed graph training

This notebook runs one authorized `model + fold` job at a time. Each job first selects `best_epoch_count` from the approved inner blocks, discards those weights, reinitializes with seed 42, refits on the complete fold-training scenarios, and only then evaluates the outer fold.

Epoch-boundary state is saved locally after every epoch and durably every five epochs. Completed jobs are immutable and normal reruns reuse them. No threshold is selected here. Test1, Test2, `train_pub_exf`, and `train_user_prop` remain prohibited.

## 1. Mount Drive and prepare a GPU runtime

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_GRAPH_PARENT = Path("/content/capture_graph_stage2_inputs_v2")
LOCAL_SELECTION_PARENT = Path("/content/capture_graph_selection_inputs")
LOCAL_WORK_ROOT = Path("/content/capture_graph_training_work")

repository_already_present = PROJECT_ROOT.exists()
if not repository_already_present:
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )
if repository_already_present:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", REPOSITORY_BRANCH],
        cwd=PROJECT_ROOT,
        check=True,
    )
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture-mlp.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
import torch
from IPython.display import display
from utils.capture_graph_dataset import stage_capture_graph_materialization
from utils.capture_graph_training import run_capture_graph_training_job
from utils.capture_graph_training_binding import (
    stage_capture_selection_materialization,
)

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime before training.")
print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("CUDA device:", torch.cuda.get_device_name(0))

## 2. Bind the authorized run

In [ ]:
GRAPH_MATERIALIZATION_RUN_ID = (
    "20260928T175821_521784Z_graph_materialization_full_dev"
)
SELECTION_MATERIALIZATION_RUN_ID = (
    "20260929T134737_484622Z_graph_selection_materialization"
)
SELECTION_MATERIALIZATION_MANIFEST_SHA256 = (
    "846d21671d6c657903427fefe665e8d1919c74859703a068a4940d02d8c09633"
)
PREPARED_RUN_ID = "20260917T235058_827743Z_prepare_full_dev"
BINDING_RUN_ID = "20260929T144234_934044Z_graph_training_binding"

GRAPH_MATERIALIZATION_DIR = (
    DRIVE_ROOT / "graph_materialization_runs" / GRAPH_MATERIALIZATION_RUN_ID
)
SELECTION_MATERIALIZATION_DIR = (
    DRIVE_ROOT / "graph_selection_materialization_runs"
    / SELECTION_MATERIALIZATION_RUN_ID
)
PREPARED_RUN_DIR = DRIVE_ROOT / "prepared_runs" / PREPARED_RUN_ID
BINDING_DIR = DRIVE_ROOT / "graph_training_binding_runs" / BINDING_RUN_ID
DURABLE_OUTPUT_ROOT = DRIVE_ROOT / "graph_training_runs" / BINDING_RUN_ID
GRAPH_INPUT_CONTRACT_PATH = (
    PROJECT_ROOT / "configs/capture_graph_stage2_input_v2.yaml"
)
SELECTION_CONFIG_PATH = (
    PROJECT_ROOT / "configs/capture_graph_selection_materialization_v1.yaml"
)
TRAINING_CONFIG_PATH = (
    PROJECT_ROOT / "configs/capture_graph_training_v1.yaml"
)

for path in (
    GRAPH_MATERIALIZATION_DIR, SELECTION_MATERIALIZATION_DIR, PREPARED_RUN_DIR,
    BINDING_DIR, GRAPH_INPUT_CONTRACT_PATH, SELECTION_CONFIG_PATH,
    TRAINING_CONFIG_PATH,
):
    if not path.exists():
        raise FileNotFoundError(f"Required input is missing: {path}")
authorization = json.loads(
    (BINDING_DIR / "graph_training_authorization.json").read_text(
        encoding="utf-8"
    )
)
if authorization.get("model_training_authorized") is not True:
    raise RuntimeError("The bound development training run is not authorized.")
print(json.dumps(authorization, indent=2))

## 3. Stage and verify graph inputs locally

In [ ]:
collection = stage_capture_graph_materialization(
    source_root=GRAPH_MATERIALIZATION_DIR,
    local_parent=LOCAL_GRAPH_PARENT,
    contract_path=GRAPH_INPUT_CONTRACT_PATH,
    expected_run_id=GRAPH_MATERIALIZATION_RUN_ID,
)
selection_root, selection_manifest = stage_capture_selection_materialization(
    source_root=SELECTION_MATERIALIZATION_DIR,
    local_parent=LOCAL_SELECTION_PARENT,
    collection=collection,
    config_path=SELECTION_CONFIG_PATH,
    expected_run_id=SELECTION_MATERIALIZATION_RUN_ID,
    expected_manifest_sha256=SELECTION_MATERIALIZATION_MANIFEST_SHA256,
)
print("Verified complete-fold graphs:", collection.root)
print("Verified selection graphs:", selection_root)

## 4. Select one model/fold job

Start with `edge_mlp__fold_A`. Its timing provides the first empirical estimate before committing a Colab session to the remaining jobs. A rerun with the same job ID resumes from the newest compatible epoch or reuses an immutable completion.

In [ ]:
job_plan = json.loads(
    (BINDING_DIR / "training_job_plan.json").read_text(encoding="utf-8")
)
display(pd.DataFrame(job_plan["jobs"]))

JOB_ID = "edge_mlp__fold_A"
RUN_TRAINING = False
selected_jobs = {item["job_id"]: item for item in job_plan["jobs"]}
if JOB_ID not in selected_jobs:
    raise ValueError(f"Unknown job ID: {JOB_ID}")
print("Selected job:", json.dumps(selected_jobs[JOB_ID], indent=2))
print("Set RUN_TRAINING=True only when this job is ready to start or resume.")

## 5. Run or resume the selected job

The first execution can run for hours. Progress is printed once per epoch. If Colab disconnects, reconnect, rerun sections 1–4 with the same `JOB_ID`, set `RUN_TRAINING=True`, and execute this cell again. Up to four completed epochs can be lost because durable recovery state is written every five epochs.

In [ ]:
if not RUN_TRAINING:
    print("Training was not started. Set RUN_TRAINING=True in section 4.")
else:
    completion = run_capture_graph_training_job(
        job_id=JOB_ID,
        binding_dir=BINDING_DIR,
        collection=collection,
        selection_root=selection_root,
        selection_manifest=selection_manifest,
        prepared_run_dir=PREPARED_RUN_DIR,
        training_config_path=TRAINING_CONFIG_PATH,
        durable_output_root=DURABLE_OUTPUT_ROOT,
        local_work_root=LOCAL_WORK_ROOT,
    )
    print(json.dumps(completion, indent=2))

## 6. Inspect a completed job

In [ ]:
completion_path = DURABLE_OUTPUT_ROOT / JOB_ID / "completion.json"
if completion_path.is_file():
    completion = json.loads(completion_path.read_text(encoding="utf-8"))
    metrics = json.loads(
        (DURABLE_OUTPUT_ROOT / JOB_ID / "metrics.json").read_text(
            encoding="utf-8"
        )
    )
    print(json.dumps(completion, indent=2))
    display(pd.DataFrame.from_dict(
        metrics["outer_oof"]["scenarios"], orient="index"
    ))
else:
    print("The selected job is not complete yet.")